# Targeted Social Prescription (Social-prescribing-inspired)
## Notebook 1: Data preprocessing & cohort construction (RAND HRS 1992–2022)

This notebook:
1) loads RAND HRS longitudinal data;
2) reshapes wave-specific variables into a long panel format;
3) constructs baseline risk factors (clinical + social needs proxies);
4) defines a wave-to-wave “~12–24 month” follow-up hospitalisation outcome (next wave).

Notes:
- We use RAND harmonised variables (prefix r*) whenever possible.
- We treat the constructed targeting logic as a transparent, rule-based eligibility algorithm.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_stata("../data_raw/randhrs1992_2022v1.dta")
print("Raw shape:", df.shape)
df.head()

In [ ]:
# RAND HRS常用个人ID：hhidpn
assert "hhidpn" in df.columns, "hhidpn not found. Please check ID column name."
df["hhidpn"].nunique(), df.shape[0]

In [ ]:
import pandas as pd
import numpy as np
import re

# ----------------------------
# Helpers
# ----------------------------
def pick_col(df: pd.DataFrame, col: str) -> pd.Series:
    """Return df[col] if exists, else a NaN series."""
    if col in df.columns:
        return df[col]
    return pd.Series([np.nan] * len(df), index=df.index)

def label_to_number(x):
    """
    Convert labeled strings like:
      '0.no' / '1.yes' / '0.no-never had cond' / '1.yes-has/had cond'
    into numeric 0/1 by extracting the leading integer.
    """
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        m = re.match(r"^\s*(-?\d+)", x)
        return float(m.group(1)) if m else np.nan
    return x

# ----------------------------
# Build long panel (hhidpn x wave) -- MATCH YOUR REAL COLUMN NAMES
# ----------------------------
def build_long_panel(df: pd.DataFrame, waves=range(1, 17)) -> pd.DataFrame:
    """
    Based on your file:
      age:      s{w}dage_y
      gender:   s1gender
      diabetes: r{w}diabe2
      ADL:      r{w}dadliv
      CESD:     r{w}cesd  (fallback r{w}cesdm)
      hosp:     r{w}hosp
    """
    out = []
    gender = pick_col(df, "s1gender")

    for w in waves:
        age_col = f"s{w}dage_y"
        diab_col = f"r{w}diabe2"
        adl_col = f"r{w}dadliv"
        hosp_col = f"r{w}hosp"

        # CESD: prefer r{w}cesd, if missing then use r{w}cesdm
        cesd_col = f"r{w}cesd" if f"r{w}cesd" in df.columns else f"r{w}cesdm"

        tmp = pd.DataFrame({
            "hhidpn": df["hhidpn"],
            "wave": w,
            "age": pick_col(df, age_col),
            "gender": gender,
            "diabetes": pick_col(df, diab_col),
            "adl_any": pick_col(df, adl_col),
            "cesd": pick_col(df, cesd_col),
            "hosp": pick_col(df, hosp_col),
        })
        out.append(tmp)

    long_df = pd.concat(out, ignore_index=True)

    # --- Convert to numeric safely ---
    long_df["age"] = pd.to_numeric(long_df["age"], errors="coerce")

    # diabetes / adl_any / hosp are often labeled categories -> extract leading number
    for c in ["diabetes", "adl_any", "hosp"]:
        long_df[c] = long_df[c].map(label_to_number)
        long_df[c] = pd.to_numeric(long_df[c], errors="coerce")

    # CESD should be numeric, but keep safe conversion
    long_df["cesd"] = pd.to_numeric(long_df["cesd"], errors="coerce")

    # --- Clean negative values (RAND missing codes often <0) ---
    for c in ["age", "diabetes", "adl_any", "cesd", "hosp"]:
        long_df.loc[long_df[c] < 0, c] = np.nan

    return long_df

# ----------------------------
# Run
# ----------------------------
panel = build_long_panel(df)

print("Panel shape:", panel.shape)
panel.head()

In [ ]:
panel.isna().mean().sort_values(ascending=False).head(10)

In [ ]:
# 关键变量：age/diabetes/adl_any/cesd/hosp 至少要有一些信息
panel["has_key_info"] = (
    panel["age"].notna() &
    (panel[["diabetes","adl_any","cesd","hosp"]].notna().any(axis=1))
)

# 候选baseline：age>=65 且有关键变量
candidates = panel[(panel["age"] >= 65) & (panel["has_key_info"])].copy()

# 每人取最早波次作为 baseline
baseline_idx = candidates.sort_values(["hhidpn", "wave"]).groupby("hhidpn").head(1).index
baseline = candidates.loc[baseline_idx].copy()

print("Baseline cohort n:", baseline["hhidpn"].nunique())
baseline.head()

In [ ]:
baseline["high_depression"] = (baseline["cesd"] >= 3).astype("float")  # 1/0/NaN
baseline["adl_limited"] = (baseline["adl_any"] == 1).astype("float")  # 假设1=有困难
baseline["has_diabetes"] = (baseline["diabetes"] == 1).astype("float")  # 假设1=有糖尿病

# 一个最小可行的 targeting / eligibility 逻辑（可在 Notebook2 再扩展）
baseline["high_risk"] = (
    (baseline["has_diabetes"] == 1) |
    (baseline["adl_limited"] == 1) |
    (baseline["high_depression"] == 1)
).astype(int)

baseline["high_risk"].value_counts(dropna=False)

In [ ]:
def quick_value_counts(s, name, n=15):
    vc = s.value_counts(dropna=False).head(n)
    print(f"\n{name} value counts (top {n}):")
    print(vc)

quick_value_counts(baseline["diabetes"], "r*diabe2 (baseline)")
quick_value_counts(baseline["adl_any"], "r*adliv (baseline)")
quick_value_counts(baseline["cesd"], "r*cesd (baseline)")
quick_value_counts(baseline["hosp"], "r*hosp (baseline)")

In [ ]:
panel = panel.sort_values(["hhidpn", "wave"])
panel["next_hosp"] = panel.groupby("hhidpn")["hosp"].shift(-1)

# 把 baseline 的下一波结局拼回来
baseline = baseline.merge(
    panel[["hhidpn", "wave", "next_hosp"]],
    on=["hhidpn", "wave"],
    how="left"
)

baseline[["hhidpn","wave","hosp","next_hosp","high_risk"]].head(10)

In [ ]:
summary = baseline.groupby("high_risk").agg(
    n=("hhidpn","nunique"),
    mean_age=("age","mean"),
    diabetes_rate=("has_diabetes","mean"),
    adl_limited_rate=("adl_limited","mean"),
    depression_rate=("high_depression","mean"),
    next_hosp_mean=("next_hosp","mean"),
)
summary

In [ ]:
import scipy.stats as stats

table = pd.crosstab(
    baseline["high_risk"],
    baseline["next_hosp"]
)

chi2, p, dof, exp = stats.chi2_contingency(table)

print("Chi-square p-value:", p)
table

In [ ]:
import statsmodels.api as sm

df_model = baseline[baseline["next_hosp"].notna()].copy()

X = df_model[["high_risk", "age"]]
X = sm.add_constant(X)

y = df_model["next_hosp"]

model = sm.Logit(y, X).fit()
print(model.summary())

In [ ]:
# ============================
# Final analytic dataset export
# ============================

final_cols = [
    "hhidpn",
    "wave",
    "age",
    "gender",
    "diabetes",
    "adl_any",
    "cesd",
    "hosp",
    "high_risk",
    "next_hosp"
]

analytic_df = baseline[final_cols].copy()

print("Final analytic dataset shape:", analytic_df.shape)

# 保存为处理后数据
analytic_df.to_csv("../data_processed/notebook1_baseline_analytic.csv", index=False)

print("Saved to data_processed/notebook1_baseline_analytic.csv")